# Rigol Oscilloscope Test Notebook

Instantiates `instruments.rigol.Rigol` (auto-detecting its USB VISA
resource), connects, configures the timebase/channels/trigger, acquires a
single-shot waveform, and demonstrates the GMM-based plateau-voltage
extraction and derived series-resistor current measurement.

**No real instrument is connected to this development machine.** Every
live-hardware call below is wrapped in `try`/`except` so this notebook
runs cleanly end-to-end and prints a clear "hardware not available"
message instead of raising, when run here. On the lab computer (with the
Rigol actually connected), the same cells should run against real
hardware.

In [ ]:
import sys
from pathlib import Path

# Make sure the project root (containing `instruments/`) is importable,
# whether this notebook is run from the repo root or from within notebooks/.
project_root = Path.cwd()
if not (project_root / "instruments").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
from instruments.rigol import Rigol, RigolError
import numpy as np

print("Rigol() with no resource argument auto-detects the single USB VISA resource.")

## Connect

In [ ]:
rigol = Rigol()
rigol_connected = False

try:
    rigol.connect()
    rigol_connected = True
    print(f"Connected to Rigol at {rigol.resource}")
except Exception as exc:
    print(f"Hardware not available: could not connect to Rigol ({exc})")

## Configure timebase, channel scales, and edge trigger

In [ ]:
try:
    if not rigol_connected:
        raise RuntimeError("Rigol not connected")
    rigol.configure_timebase(1e-5, offset=50e-6)
    rigol.configure_channel(1, 2.0)
    rigol.configure_channel(2, 2.0)
    rigol.configure_channel(3, 2.0)
    rigol.configure_trigger(3, level=2.0, slope="POSITIVE")
    print("Timebase, channel scales, and edge trigger configured.")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## Single-shot acquisition

In [ ]:
try:
    if not rigol_connected:
        raise RuntimeError("Rigol not connected")
    waveforms = rigol.acquire_single_shot([1, 2])
    for channel, (t, v) in waveforms.items():
        print(f"Channel {channel}: {len(v)} samples, mean={np.mean(v):.4f} V")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## GMM plateau extraction (no hardware needed)

`extract_plateau_voltage` is pure signal processing -- it fits a
2-component Gaussian Mixture Model to a waveform and returns the higher
mean as the pulse-plateau value. This works regardless of whether hardware
is connected, so it is demonstrated here on clearly-labeled synthetic
`dummy_data` rather than a real waveform.

In [ ]:
rng = np.random.default_rng(0)
dummy_data = np.concatenate(
    [
        rng.normal(0.0, 0.05, 800),  # synthetic baseline level
        rng.normal(3.0, 0.05, 200),  # synthetic pulse plateau level
    ]
)
plateau_voltage = rigol.extract_plateau_voltage(dummy_data)
print(f"Extracted plateau voltage from synthetic dummy_data: {plateau_voltage:.4f} V")

## Plateau voltage and derived current measurement (needs hardware)

In [ ]:
try:
    if not rigol_connected:
        raise RuntimeError("Rigol not connected")
    plateau = rigol.measure_plateau_voltage(1)
    print(f"Measured plateau voltage on channel 1: {plateau:.4f} V")
    current = rigol.measure_series_resistor_current(
        upstream_channel=2, downstream_channel=1, series_resistance_ohm=50.0
    )
    print(f"Derived current across series resistor: {current:.6f} A")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## Disconnect

In [ ]:
try:
    if rigol_connected:
        rigol.disconnect()
        print("Disconnected from Rigol.")
    else:
        print("Skipping disconnect: Rigol was never connected.")
except Exception as exc:
    print(f"Error during disconnect: {exc}")

## Context manager usage

In [ ]:
try:
    with Rigol() as scope:
        print(f"Connected via context manager at {scope.resource}")
except Exception as exc:
    print(f"Hardware not available: {exc}")